# 06. Interval DP and Partition DP

The state is a **range** `dp[l][r]`. We try every **split point** `k` inside the range. Fill ranges in order of increasing length.

**Learning order:** IDEA → BURST BALLOONS → MATRIX CHAIN → MIN PALINDROME CUTS → INTERVIEW THINKING

## 1. Idea

Problems like "best way to cut/merge/burst a sequence". Small ranges first, then bigger ranges combine smaller ones.

```
for length in range(2, n + 1):
    for l in range(0, n - length + 1):
        r = l + length - 1
        for k in range(l, r):               # split point
            dp[l][r] = best(dp[l][r], dp[l][k] + dp[k+1][r] + cost)
```

Time O(n³) usually.

## 2. Burst Balloons

Bursting balloon `i` gives `nums[i-1] * nums[i] * nums[i+1]` coins. Maximize the total.

**Key trick:** think of the **last** balloon to burst in a range `(l, r)`, not the first. If `k` is last, the balloons to its left and right are already gone, so its neighbours are exactly `l` and `r` (independent subproblems).

`dp[l][r] = max over k: dp[l][k] + dp[k][r] + nums[l] * nums[k] * nums[r]` (exclusive range `l < k < r`, padded with `1` at both ends).

In [1]:
def max_coins(nums):
    a = [1] + nums + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):                         # distance between l and r
        for l in range(0, n - length):
            r = l + length
            for k in range(l + 1, r):                  # k = last balloon burst in (l, r)
                dp[l][r] = max(dp[l][r], dp[l][k] + dp[k][r] + a[l] * a[k] * a[r])
    return dp[0][n - 1]

print(max_coins([3, 1, 5, 8]), max_coins([1, 5]))   # 167 10

167 10


## 3. Matrix Chain Multiplication

Multiplying matrices `A1..An` where `Ai` has size `dims[i-1] × dims[i]`. The **order of brackets** changes the number of multiplications. Find the minimum.

`dp[i][j]` = min cost to multiply `Ai..Aj`: try every split `k`: `dp[i][k] + dp[k+1][j] + dims[i-1] * dims[k] * dims[j]`.

In [2]:
def matrix_chain_order(dims):
    n = len(dims) - 1                                  # number of matrices
    dp = [[0] * (n + 1) for _ in range(n + 1)]
    for length in range(2, n + 1):
        for i in range(1, n - length + 2):
            j = i + length - 1
            dp[i][j] = float("inf")
            for k in range(i, j):
                cost = dp[i][k] + dp[k + 1][j] + dims[i - 1] * dims[k] * dims[j]
                dp[i][j] = min(dp[i][j], cost)
    return dp[1][n]

print(matrix_chain_order([10, 30, 5, 60]))   # 4500

4500


## 4. Palindrome Partitioning II (minimum cuts)

Cut a string into palindromes with the **fewest cuts**.

1. Precompute `is_pal[i][j]` with DP (a substring is a palindrome if the ends match and the inside is a palindrome).
2. `cuts[i]` = min cuts for `s[:i+1]`: if the whole prefix is a palindrome, `0`; else `min(cuts[j] + 1)` for every `j` where `s[j+1..i]` is a palindrome.

In [3]:
def min_cut(s):
    n = len(s)
    is_pal = [[False] * n for _ in range(n)]
    for i in range(n - 1, -1, -1):
        for j in range(i, n):
            if s[i] == s[j] and (j - i < 2 or is_pal[i + 1][j - 1]):
                is_pal[i][j] = True
    cuts = [0] * n
    for i in range(n):
        if is_pal[0][i]:
            cuts[i] = 0
        else:
            cuts[i] = min(cuts[j] + 1 for j in range(i) if is_pal[j + 1][i])
    return cuts[-1]

print(min_cut("aab"), min_cut("a"), min_cut("ab"))   # 1 0 1

1 0 1


## 5. Interview thinking

**Signals:** "burst / merge / split a sequence", "best bracket order", "minimum cost to cut", "partition into palindromes".

**Say:** "The state is the subarray `[l, r]`. I try each split `k` and combine the two smaller ranges. I fill by increasing length. O(n³) time, O(n²) space."

**Common mistakes:** thinking about the **first** instead of the **last** operation (Burst Balloons), wrong loop order (must be by increasing length), padding forgotten, off-by-one in inclusive/exclusive ranges.

## Quick Revision

- `dp[l][r]` over ranges, loop by increasing length, try every split `k`.
- Burst balloons: choose the **last** one to burst.
- Matrix chain: `dp[i][k] + dp[k+1][j] + dims[i-1]*dims[k]*dims[j]`.
- Palindrome cuts: precompute `is_pal`, then 1D min over valid last pieces.